In [1]:
import pandas as pd
from libreco.data import DatasetPure
from libreco.algorithms import UserCF

Instructions for updating:
non-resource variables are not supported in the long term


In [6]:
items = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/books_data_with_new_id.csv')

train_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/no dup new/amazon_books_ratings_train_filtered_fixed.csv')
val_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/no dup new/amazon_books_ratings_val_filtered_fixed.csv')
test_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/no dup new/amazon_books_ratings_test_filtered_fixed.csv')

In [8]:
train_ratings

,Unnamed: 0.1,Unnamed: 0,item_id,Title,user_id,profileName,rating,timestamp
0,192634,2808945,B000NWR0W6,and ladies of the club,A3RTKL9KB8KLID,Stan Vernooy,5.0,854150400
1,192637,2144019,B000MVVYNO,Bethlehem Road,A3RTKL9KB8KLID,Stan Vernooy,4.0,862358400
2,195879,2520883,B000HEV3WS,"HERCULES, MY SHIPMATE",A3TEH90X39WC8F,"Stuart W. Mirsky ""swm""",5.0,863308800
3,195797,2196179,B0007E212E,"Hercules, My Shipmate",A3TEH90X39WC8F,"Stuart W. Mirsky ""swm""",5.0,863308800
4,238620,346828,B000KISQC6,"Guns, Germs, and Steel: The Fates of Human Soc...",AMGVCPHMT1DWE,Gregory N. Hullender,5.0,865468800
...,...,...,...,...,...,...,...,...
262291,96431,1558301,1593355548,Wuthering Heights,A2DKTZMMG3JHN4,K. Burns,4.0,1362268800
262292,96428,1343379,1569602093,Wuthering Heights.,A2DKTZMMG3JHN4,K. Burns,4.0,1362268800
262293,96420,861842,1593355548,Wuthering Heights,A2DKTZMMG3JHN4,K. Burns,4.0,1362268800
262294,103760,2616778,1844560333,Pride and Prejudice,A2GDT5QQSFZD14,Joy Hilda Handley,5.0,1362268800


In [10]:
# Rename columns to match the expected format
train_ratings.rename(columns={'user_id': 'user', 'item_id': 'item', 'rating': 'label', 'timestamp': 'time'}, inplace=True)
val_ratings.rename(columns={'user_id': 'user', 'item_id': 'item', 'rating': 'label', 'timestamp': 'time'}, inplace=True)
test_ratings.rename(columns={'user_id': 'user', 'item_id': 'item', 'rating': 'label', 'timestamp': 'time'}, inplace=True)

# Ensure the columns are in the correct order
train_ratings = train_ratings[['user', 'item', 'label', 'time']]
val_ratings = val_ratings[['user', 'item', 'label', 'time']]
test_ratings = test_ratings[['user', 'item', 'label', 'time']]

In [11]:
train_data, data_info = DatasetPure.build_trainset(train_ratings)
eval_data = DatasetPure.build_evalset(val_ratings)
test_data = DatasetPure.build_testset(test_ratings)
print(data_info)  # n_users: 5894, n_items: 3253, data sparsity: 0.4172 %

n_users: 6318, n_items: 45862, data density: 0.0905 %


In [12]:
train_ratings = train_ratings.sort_values(by='time')
val_ratings = val_ratings.sort_values(by='time')
test_ratings = test_ratings.sort_values(by='time')

In [108]:
user_cf = UserCF(task="ranking", data_info=data_info, k_sim=100, sim_type="cosine", mode='invert')

In [109]:
# Training the model
user_cf.fit(train_data, verbose=2, eval_data=eval_data, k=5, metrics=["loss", "roc_auc", "precision", "recall", "ndcg"], neg_sampling=True)

Training start time: 2024-09-01 21:13:38
Final block size and num: (6318, 1)
sim_matrix elapsed: 0.372s
sim_matrix, shape: (6318, 6318), num_elements: 4289274, density: 10.7454 %


eval_pointwise:   0%|          | 0/8 [00:00<?, ?it/s]

Detect 353 unknown interaction(s), position: [2064, 7186, 7188, 1048, 30, 4134, 1066, 3116, 6190, 1074, 7218, 4150, 2110, 4160, 4162, 4164, 3142, 1096, 2122, 5200, 4088, 4188, 1120, 5216, 7264, 6244, 104, 3176, 106, 3180, 3182, 1140, 126, 1154, 7300, 6278, 7306, 140, 2188, 1164, 4242, 4246, 5278, 6306, 4262, 6310, 2216, 6312, 170, 5286, 3244, 3248, 180, 7350, 186, 7354, 188, 4284, 2238, 7360, 2244, 4292, 4294, 6340, 7372, 2256, 6352, 5332, 4314, 7386, 4318, 3294, 8186, 6372, 7396, 3304, 7400, 3310, 242, 6388, 5368, 5370, 4352, 262, 3342, 4370, 6418, 7444, 280, 5402, 3362, 7458, 3368, 304, 7472, 2354, 7480, 2362, 6460, 5438, 4416, 5442, 326, 1356, 6482, 4436, 344, 2392, 6490, 2398, 7520, 1388, 366, 7536, 3456, 4492, 2450, 2454, 408, 3480, 4506, 3482, 7578, 2462, 7582, 6562, 7588, 6566, 3494, 424, 426, 3498, 6572, 5546, 430, 7596, 2480, 3510, 2490, 2496, 7616, 6594, 5572, 7622, 4554, 5580, 462, 1488, 5584, 468, 6612, 470, 2520, 2522, 3546, 6626, 2532, 2534, 4582, 5608, 7658, 4590, 7670, 

eval_pointwise:  12%|█▎        | 1/8 [00:00<00:02,  2.98it/s]

Detect 342 unknown interaction(s), position: [2, 12, 5132, 4110, 22, 26, 7200, 38, 5158, 2092, 6188, 2096, 3120, 7232, 1094, 2124, 7244, 4174, 6222, 6224, 3152, 5208, 2138, 5214, 2144, 1120, 98, 7264, 7266, 1124, 4200, 6248, 6250, 1130, 2156, 1132, 110, 1134, 2160, 6256, 6260, 7284, 6264, 3196, 126, 2178, 3206, 7306, 7310, 3216, 1172, 3220, 4246, 1174, 4248, 1180, 5276, 3236, 7334, 172, 5292, 3248, 2228, 1204, 4280, 6330, 2236, 1212, 2244, 5316, 5320, 6346, 3274, 4300, 7372, 1230, 3280, 5328, 3292, 5340, 5342, 2272, 2278, 3302, 4336, 2294, 6390, 248, 6394, 7422, 6402, 1282, 3334, 4360, 1290, 6412, 7438, 1298, 5394, 5396, 1302, 3350, 5400, 282, 1312, 2338, 1316, 2342, 1332, 7480, 7486, 326, 1350, 3406, 2384, 338, 2386, 3410, 6492, 5468, 1374, 5472, 354, 1380, 358, 6504, 1384, 1388, 4462, 1390, 370, 2418, 1394, 374, 5494, 3450, 5508, 392, 5512, 5514, 4492, 5516, 3472, 4498, 5526, 3488, 2474, 4524, 5548, 2478, 7596, 6578, 436, 6580, 3528, 5584, 5588, 4566, 6616, 5596, 2526, 3550, 4576, 76

...

Detect 337 unknown interaction(s), position: [2054, 6152, 7176, 5132, 7190, 6180, 4134, 3112, 2096, 6192, 2100, 58, 3130, 7228, 5184, 3142, 2120, 3146, 6222, 7246, 6234, 92, 5216, 5218, 102, 4198, 4200, 5224, 1130, 3178, 5236, 124, 128, 1154, 5250, 134, 4230, 4232, 138, 4234, 2194, 3226, 2204, 7328, 7330, 5284, 6318, 6142, 3258, 7354, 2236, 5314, 3268, 5316, 6344, 2252, 7372, 6350, 4308, 5332, 4312, 2272, 6372, 1254, 7402, 242, 2290, 1266, 3316, 1272, 250, 5370, 2300, 5372, 254, 3326, 256, 7422, 3328, 2308, 1284, 1286, 4360, 5384, 4362, 270, 7438, 6416, 2334, 4384, 1312, 1316, 7460, 5414, 1320, 1322, 5420, 302, 7470, 306, 7484, 6466, 3398, 4424, 2382, 7502, 4438, 3414, 7520, 356, 6502, 370, 6516, 7540, 2422, 3448, 1404, 384, 7554, 2436, 3460, 4486, 5510, 5512, 1418, 7572, 1432, 2458, 6556, 4510, 2464, 4514, 2468, 7592, 1454, 432, 4534, 5558, 1468, 2494, 1470, 6594, 6602, 2508, 5582, 3544, 7644, 7646, 7650, 2532, 5612, 7662, 4592, 1520, 7664, 1522, 5620, 4600, 1530, 7676, 512, 5632, 768

eval_pointwise:  88%|████████▊ | 7/8 [00:02<00:00,  3.06it/s]

Detect 308 unknown interaction(s), position: [4100, 1032, 6154, 4108, 14, 1044, 2074, 4122, 7198, 6190, 4146, 5172, 54, 3126, 3132, 2116, 3142, 6220, 2126, 4178, 86, 1110, 5206, 6236, 2148, 4196, 2150, 1130, 2156, 3180, 112, 3186, 5234, 2164, 1142, 1144, 4218, 130, 2180, 6278, 1158, 5254, 4240, 2196, 2206, 2208, 5280, 162, 168, 6312, 5288, 6326, 3254, 2232, 4280, 188, 190, 6336, 1216, 196, 200, 202, 2250, 6346, 5324, 6352, 3280, 218, 4320, 5344, 3300, 230, 6374, 232, 2280, 1254, 3304, 5354, 1262, 1266, 5364, 3318, 5370, 2302, 5376, 2314, 268, 4370, 6422, 4378, 6428, 7452, 2334, 7454, 4386, 6434, 1314, 1316, 294, 4390, 7460, 3368, 2350, 306, 2360, 6456, 1338, 1346, 5446, 1356, 4434, 7506, 4438, 5482, 1390, 3440, 1394, 3444, 378, 4474, 4476, 7548, 4478, 2434, 388, 3468, 1422, 5518, 400, 2448, 3486, 4520, 3498, 434, 6578, 2484, 7604, 3510, 1464, 2494, 3518, 454, 4550, 3530, 3532, 5584, 2520, 3544, 500, 3572, 4604, 4608, 7680, 3588, 7684, 1548, 528, 4624, 1552, 536, 4632, 7706, 6688, 3618,

eval_listwise: 100%|██████████| 6318/6318 [00:49<00:00, 128.85it/s]


	 eval log_loss: 7.2221
	 eval roc_auc: 0.7897
	 eval precision@5: 0.1103
	 eval recall@5: 0.1461
	 eval ndcg@5: 0.3448


In [110]:
from libreco.evaluation import evaluate

# Evaluate the model on the test data with the specified metrics
evaluation_results = evaluate(
    model=user_cf,
    data=test_data,
    neg_sampling=True,
    metrics=["loss", "roc_auc", "precision", "recall", "ndcg"]
)

# Print the evaluation results
for metric, value in evaluation_results.items():
    print(f"{metric}: {value}")

eval_pointwise:   0%|          | 0/9 [00:00<?, ?it/s]

Detect 310 unknown interaction(s), position: [0, 2, 1026, 2052, 6148, 6154, 7180, 6162, 22, 7194, 2076, 4126, 1054, 3102, 5158, 7212, 56, 5176, 7228, 3136, 7236, 6214, 5190, 2120, 7240, 2122, 4170, 80, 3154, 4182, 1114, 4188, 7282, 5242, 2172, 4222, 4224, 4228, 6278, 3212, 4246, 5118, 4256, 6304, 3234, 7332, 4266, 7338, 4274, 7350, 6328, 7352, 1218, 3266, 5314, 2246, 6342, 7376, 7166, 7382, 3290, 3298, 4332, 6384, 6388, 1278, 3326, 4368, 6416, 1300, 4374, 6422, 5398, 6430, 6432, 4390, 298, 6446, 1328, 7472, 1334, 3384, 5432, 4412, 6460, 6462, 3392, 3400, 7504, 4434, 2394, 6490, 5466, 5476, 2406, 1382, 2410, 2412, 6508, 2414, 2418, 2424, 3448, 3450, 4478, 3456, 3464, 2442, 5514, 6540, 6542, 7570, 3474, 5522, 6550, 408, 4504, 4508, 414, 6558, 7582, 1442, 3498, 5548, 2480, 6576, 3504, 7600, 3508, 2486, 440, 3512, 5560, 2492, 6590, 3518, 6594, 456, 2518, 472, 2532, 486, 3558, 2536, 3560, 7656, 1516, 494, 7670, 1530, 1532, 1534, 5636, 6662, 7690, 6668, 1548, 1550, 2578, 1556, 2584, 7704, 53

eval_pointwise:  11%|█         | 1/9 [00:00<00:02,  2.78it/s]

Detect 330 unknown interaction(s), position: [3072, 1028, 8, 26, 4122, 2076, 1058, 2088, 4136, 1066, 3114, 44, 3116, 2096, 4146, 1074, 6196, 4156, 6206, 3134, 6210, 3138, 4164, 6212, 74, 4170, 5194, 2128, 7254, 4184, 94, 98, 3170, 4204, 4206, 1134, 112, 7282, 4212, 6262, 3194, 7292, 5246, 3202, 1158, 142, 3216, 6294, 1176, 5282, 166, 6314, 1194, 172, 6316, 3254, 5312, 3266, 7364, 6344, 6348, 3278, 6352, 3280, 4310, 7382, 3288, 2268, 4316, 1244, 226, 232, 7404, 240, 6384, 242, 7408, 248, 4354, 6404, 2312, 270, 6426, 2332, 1310, 1314, 4390, 4394, 302, 3378, 308, 4404, 6454, 4406, 312, 3382, 314, 4410, 6458, 6460, 318, 3388, 3392, 322, 2388, 2390, 3414, 1370, 3418, 1374, 2400, 6496, 7526, 3432, 3434, 5484, 2418, 1396, 5502, 2434, 2446, 3474, 7570, 7572, 6552, 7578, 412, 2462, 416, 7586, 4516, 7588, 2476, 5554, 438, 5560, 4538, 5562, 2492, 3516, 446, 1470, 5568, 7616, 2502, 5576, 7624, 6604, 1486, 5582, 1490, 7636, 1494, 4578, 1514, 492, 7664, 2546, 4596, 5624, 6650, 3578, 2556, 1538, 6660

eval_pointwise:  22%|██▏       | 2/9 [00:00<00:02,  2.88it/s]

...

Detect 368 unknown interaction(s), position: [1024, 1026, 2052, 4100, 2054, 5124, 8, 2058, 6154, 3086, 2078, 5150, 36, 6182, 1062, 2088, 7206, 5160, 1066, 4142, 1070, 7218, 54, 7224, 4154, 3134, 64, 6208, 5186, 2116, 2120, 6224, 3154, 1114, 1126, 6248, 1130, 2156, 6254, 6260, 6262, 1144, 3192, 3194, 3204, 3206, 4232, 3208, 4234, 6282, 5256, 4242, 6292, 4246, 1174, 1176, 3224, 7322, 5276, 7324, 4258, 5286, 2216, 5292, 6320, 6322, 2232, 1218, 4292, 7364, 4296, 1224, 1228, 1230, 6354, 2260, 3288, 6364, 5344, 6370, 6372, 1254, 6376, 238, 6382, 4340, 5364, 5368, 7422, 5376, 4354, 6406, 7432, 4374, 7446, 7448, 2334, 4382, 3358, 5408, 1314, 7462, 6446, 5424, 3378, 310, 5430, 7488, 7498, 332, 6478, 4436, 1364, 344, 2394, 1370, 5470, 7520, 7522, 3430, 7526, 360, 2410, 6508, 3438, 1392, 6518, 3446, 376, 2428, 6524, 3460, 5508, 6538, 5514, 4494, 7570, 2452, 1432, 412, 4514, 2468, 422, 7590, 2472, 1452, 430, 6574, 432, 434, 2484, 3508, 2486, 6582, 3510, 1466, 5564, 1470, 7614, 2504, 6606, 3538, 76

eval_pointwise:  89%|████████▉ | 8/9 [00:02<00:00,  2.94it/s]

Detect 198 unknown interaction(s), position: [1024, 8, 520, 1546, 6156, 14, 1556, 6170, 3102, 6174, 34, 2082, 1060, 4132, 1062, 5670, 552, 2600, 3112, 3626, 556, 2092, 558, 4136, 2610, 3122, 1588, 2100, 566, 2614, 3064, 4664, 3130, 4666, 2112, 3066, 68, 4166, 2120, 4170, 2124, 594, 596, 1624, 6232, 2654, 2144, 6240, 1634, 614, 2664, 4712, 1130, 4092, 2162, 2166, 4094, 2680, 4726, 4218, 1670, 6278, 1166, 144, 1170, 2200, 2202, 1180, 2204, 670, 4252, 1190, 2216, 2218, 172, 3756, 1198, 4270, 178, 2228, 2234, 2746, 1724, 190, 4286, 4290, 1220, 4296, 4814, 5330, 212, 5334, 6360, 1244, 5342, 4330, 240, 4850, 2294, 3832, 5882, 2812, 4348, 2816, 5890, 2308, 3844, 3334, 3846, 1288, 4362, 268, 2320, 280, 288, 2348, 4912, 830, 1856, 326, 1350, 328, 2886, 1868, 3404, 1872, 1878, 3414, 1368, 3416, 1370, 4440, 1884, 1886, 4446, 352, 6494, 1892, 6500, 358, 1894, 872, 3430, 2922, 1900, 6178, 2928, 884, 6008, 890, 3452, 2432, 386, 898, 1412, 5508, 1928, 2448, 3474, 1946, 5530, 1954, 3492, 2470, 4010, 4

eval_listwise: 100%|██████████| 6318/6318 [00:47<00:00, 132.13it/s]


loss: 6.968312536452403
roc_auc: 0.797481547844862
precision: 0.0750079138968028
recall: 0.1747051791035233
ndcg: 0.4083102389948184


In [116]:
import numpy as np
def dcg(scores, k):
    scores = np.asfarray(scores)[:k]
    return np.sum(scores / np.log2(np.arange(2, scores.size + 2)))

def ndcg_at_k(labels, k):
    ideal_labels = sorted(labels, reverse=True)
    return dcg(labels, k) / dcg(ideal_labels, k)

def recall_at_k(labels, relevant_count, k):
    return np.sum(labels[:k]) / relevant_count

def mrr_at_k(labels, k):
    for i, label in enumerate(labels[:k]):
        if label == 1:
            return 1 / (i + 1)
    return 0

def evaluate_user_cf_model(model, test_data, train_data, all_items, k):
    ndcg_scores = []
    recall_scores = []
    mrr_scores = []
    iteration_counter = 0

    # Get unique users
    unique_users = test_data['user_id'].unique()

    for user in unique_users:

        # Recommend items for the user using the model|
        recommended_items = model.recommend_user(user, n_rec=k, filter_consumed=True)
        recommended_items = recommended_items[user]
        user_test_data = test_data[test_data['user_id'] == user]
        test_items = user_test_data['item_id'].values

        y_score = [1 if item in test_items else 0 for item in recommended_items]
        ndcg = ndcg_at_k(y_score, k)
        recall = recall_at_k(y_score, len(test_items), k)
        mrr = mrr_at_k(y_score, k)

        ndcg_scores.append(ndcg)
        recall_scores.append(recall)
        mrr_scores.append(mrr)
        iteration_counter += 1

    # avg_ndcg = np.mean(np.nan_to_num(ndcg_scores, nan=0.0))

    avg_ndcg = np.nanmean(ndcg_scores)
    avg_recall = np.nanmean(recall_scores)
    avg_mrr = np.nanmean(mrr_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg,
        'Recall@{}'.format(k): avg_recall,
        'MRR@{}'.format(k): avg_mrr,
    }

# all_items = movies['item_id'].unique()
all_items = items['item_id'].unique()

# Evaluate the model
eval_result = evaluate_user_cf_model(user_cf, test_ratings, train_ratings, all_items, k=5)
print(eval_result)
eval_result = evaluate_user_cf_model(user_cf, test_ratings, train_ratings, all_items, k=10)
print(eval_result)

C:\Users\Hooman\AppData\Local\Temp\ipykernel_11928\496216468.py:8: RuntimeWarning: invalid value encountered in scalar divide
  return dcg(labels, k) / dcg(ideal_labels, k)


{'NDCG@5': 0.8471920717470142, 'Recall@5': 0.1478280083141099, 'MRR@5': 0.3757148886778516}
{'NDCG@10': 0.7962378973290587, 'Recall@10': 0.16709953364319655, 'MRR@10': 0.38143154856117817}


In [112]:
cf_recommendations = {}

# Assume unique_users is a list or array of all user IDs
for user in test_ratings['user_id'].unique():
    # Generate top N recommendations for the user using the Collaborative Filtering model
    recommended_items = user_cf.recommend_user(user, n_rec=5, filter_consumed=True)
    cf_recommendations[user] = recommended_items[user]  # Store the recommendations in the dictionary
cf_recommendations

{'AZUNT3QP2CWTL': array(['B0007HUT02', 'B000GKURY8', 'B0007DRGI4', 'B000N3ZGB2',
        'B000NWUHR6'], dtype='<U10'),
 'A3VVDE8I22IAJA': array(['1844560333', '8188280046', '1593355548', '1569602093',
        'B000P4Q3JS'], dtype='<U10'),
 'A3AZ4O4I9S4668': array(['B000QB9ZYA', 'B0006AP72A', '1597374555', 'B000K0H0OO',
        'B0001GDR3U'], dtype='<U10'),
 'A319KYEIAZ3SON': array(['1844560333', '1901768600', 'B000HEGYW2', 'B000RAZERW',
        '1587263971'], dtype='<U10'),
 'A3SOB0CMUBK6XJ': array(['B0006BV6RY', '1844560333', '8188280046', '1587263971',
        '1847022251'], dtype='<U10'),
 'A3927BH5H75LII': array(['B000I3NFKG', 'B000EANQJ8', 'B0007C10MS', 'B0007GZPJI',
        'B0006AU8K6'], dtype='<U10'),
 'A2FR8GG77M4TP7': array(['B000HKLROQ', 'B00007K45C', 'B000P0W8K0', 'B000TKO3EA',
        'B000HZ9A2W'], dtype='<U10'),
 'A2YUZKPLUYQDKV': array(['B000L4056E', 'B000NDSX6C', 'B000Q032UY', 'B000ILIJE0',
        'B000NWU3I4'], dtype='<U10'),
 'A1HLJAZ1J5MMAB': array(['B000KXXBXA', '

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
import pytorch_lightning as pl
from sentence_transformers import SentenceTransformer
import numpy as np
from pytorch_lightning.callbacks import Callback
import pandas as pd

D:\Anaconda\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
D:\Anaconda\lib\site-packages\transformers\utils\generic.py:311: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(


In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.get_device_name(0)

'NVIDIA GeForce RTX 3060 Laptop GPU'

In [31]:
items = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/books_data_with_new_id.csv')

full_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/no dup new/amazon_books_ratings_full_filtered_fixed.csv')
train_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/no dup new/amazon_books_ratings_train_filtered_fixed.csv')
val_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/no dup new/amazon_books_ratings_val_filtered_fixed.csv')
test_ratings = pd.read_csv('./processed_dataset/Amazon-Books/20 interactions fixed/no dup new/amazon_books_ratings_test_filtered_fixed.csv')

In [32]:
items = items.dropna(subset=['item_id'])


In [40]:
def generate_last_user_texts_with_history(items, val_ratings):
    user_histories = {}
    last_user_texts = {}

    # Convert items to a dictionary for faster access
    items_dict = items.set_index('item_id')[['Title', 'categories', 'authors']].to_dict('index')

    # Process the val_ratings
    for _, row in val_ratings.iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        profile_name = row['profileName']  # Get the profile name directly from the ratings DataFrame

        # Initialize user history if not already done
        if user_id not in user_histories:
            user_histories[user_id] = []

        # Generate the user's history (only the last 3 items)
        history_items = []
        for mid in user_histories[user_id][-3:]:
            if mid in items_dict:
                title = items_dict[mid].get('Title', '').strip()
                category = items_dict[mid]['categories']

                # Clean and format the category, remove NaN
                if pd.notna(category):
                    category = category.strip("[]'\"")
                    if category:  # Only add non-empty, non-NaN categories
                        history_items.append(category)

        history_str = ", ".join(history_items)

        # Combine history into the final text format
        if history_str:
            combined_features = f"profileName: {profile_name} [SEP] category: {history_str}"
        else:
            combined_features = f"profileName: {profile_name}"

        # Update the dictionary to keep the last text for each user
        last_user_texts[user_id] = combined_features

        # Update the user history after generating combined features
        if item_id in items_dict:  # Ensure item exists in the dictionary
            user_histories[user_id].append(item_id)

    return last_user_texts

# Generate the last user texts for the validation data
val_last_user_texts = generate_last_user_texts_with_history(items, val_ratings)

In [43]:
items['categories'].fillna('', inplace=True)
items['cleaned_categories'] = items['categories'].str.strip('[]').str.replace("'", '')

items['book_features'] = items.apply(
    lambda row: (
        f"title: {row['Title']} [SEP] category: {row['cleaned_categories']}"
        if row['cleaned_categories'] else
        f"title: {row['Title']}"
    ),
    axis=1
)

In [45]:
# Create a dictionary for fast lookup
item_features_dict = items.set_index('item_id')['book_features'].to_dict()

# Create lists of user and item texts
item_texts = [item_features_dict[itemId] for itemId in full_ratings['item_id'].unique()]

# Create a mapping from userId and movieId to indices
item_id_to_idx = {itemId: idx for idx, itemId in enumerate(full_ratings['item_id'].unique())}

# Map userId and movieId in ratings_df to indices
train_ratings['item_idx'] = train_ratings['item_id'].map(item_id_to_idx)

# Map userId and movieId in ratings_val to indices
val_ratings['item_idx'] = val_ratings['item_id'].map(item_id_to_idx)

# Map userId and movieId in ratings_test to indices
test_ratings['item_idx'] = test_ratings['item_id'].map(item_id_to_idx)

# Extract user indices, item indices, and ratings
train_item_indices = torch.LongTensor(train_ratings['item_idx'].values).to(device)
train_labels = torch.FloatTensor(train_ratings['rating'].values).to(device)

# Extract user indices, item indices, and ratings for validation
val_item_indices = torch.LongTensor(val_ratings['item_idx'].values).to(device)
val_labels = torch.FloatTensor(val_ratings['rating'].values).to(device)

# Extract user indices, item indices, and ratings for test
test_item_indices = torch.LongTensor(test_ratings['item_idx'].values).to(device)
test_labels = torch.FloatTensor(test_ratings['rating'].values).to(device)

In [47]:
class TwoTowerModel(pl.LightningModule):
    def __init__(self, user_model_name, item_model_name, embedding_size=384):
        super(TwoTowerModel, self).__init__()
        self.user_model = SentenceTransformer(user_model_name)
        self.item_model = SentenceTransformer(item_model_name)

        self.user_fc = nn.Linear(embedding_size, embedding_size)
        self.item_fc = nn.Linear(embedding_size, embedding_size)

        self.criterion = nn.MSELoss()
        self.epoch_losses = {'train_loss': [], 'val_loss': []}

    def forward(self, user_text, item_text):
        user_embedding = self.user_model.encode(user_text, convert_to_tensor=True).to(device)
        item_embedding = self.item_model.encode(item_text, convert_to_tensor=True).to(device)

        user_output = self.user_fc(user_embedding)
        item_output = self.item_fc(item_embedding)

        dot_product = torch.matmul(user_output.squeeze(), item_output.T)
        dot_product = 4 * torch.sigmoid(dot_product) + 1

        return dot_product

    def training_step(self, batch, batch_idx):
        users, items, ratings = batch

        items = [item_texts[i] for i in items.tolist()]

        preds = self(users, items)

        loss = self.criterion(preds, ratings)
        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        users, items, ratings = batch

        items = [item_texts[i] for i in items.tolist()]

        preds = self(users, items)

        loss = self.criterion(preds, ratings)
        self.log('val_loss', loss)
        return loss

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=1e-5)

class PrintLossesCallback(Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        train_loss = trainer.callback_metrics.get('train_loss')
        if train_loss is not None:
            pl_module.epoch_losses['train_loss'].append(train_loss.item())
            print(f"Epoch {trainer.current_epoch + 1}: Train Loss: {train_loss.item()}")

    def on_validation_epoch_end(self, trainer, pl_module):
        val_loss = trainer.callback_metrics.get('val_loss')
        if val_loss is not None:
            pl_module.epoch_losses['val_loss'].append(val_loss.item())
            print(f"Epoch {trainer.current_epoch + 1}: Val Loss: {val_loss.item()}")

In [48]:
best_model_path = './lightning_logs/books/paraphrase-MiniLM-L12-v2/not-binarized/history_5-epochs-1e-5_(pron + last 3 (cat)) + books (title + cat) (no header tag)/checkpoints/epoch=4-step=20490.ckpt'

# best_model = TwoTowerModel.load_from_checkpoint(best_model_path, user_model_name='paraphrase-MiniLM-L6-v2', item_model_name='paraphrase-MiniLM-L6-v2').to(device)
best_model = TwoTowerModel.load_from_checkpoint(best_model_path, user_model_name='paraphrase-MiniLM-L12-v2', item_model_name='paraphrase-MiniLM-L12-v2').to(device)


D:\Anaconda\lib\site-packages\transformers\utils\generic.py:311: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(


In [49]:
# Assuming full_items_embeddings is already defined
full_items_embeddings = torch.stack([best_model.item_model.encode(item_text, convert_to_tensor=True) for item_text in item_texts]).to(device)

In [50]:
def get_top_n_items_without_history_unseen_items(model, userId, n):
    # Ensure the model is in evaluation mode
    model.eval()

    # Get the user text for the given userId
    user_text = val_last_user_texts[userId]
    # print(user_text)
    # Encode the user text
    user_embedding = model.user_model.encode(user_text, convert_to_tensor=True).to(device)

    # Compute the scores (dot product between user embedding and each item embedding)
    user_output = model.user_fc(user_embedding).to(device)
    item_output = model.item_fc(full_items_embeddings).to(device)
    dot_product = torch.matmul(user_output, item_output.t()).squeeze()

    # Get items the user has seen in the training and validation data
    seen_items_train = train_ratings[train_ratings['user_id'] == userId]['item_id'].values
    seen_items_val = val_ratings[val_ratings['user_id'] == userId]['item_id'].values
    seen_items = set(np.concatenate((seen_items_train, seen_items_val)))
    # print(dot_product)
    # print(len(dot_product), len(seen_items))
    # Get the top n + len(seen_items) item indices and their scores
    # top_n_scores, top_n_indices = torch.topk(dot_product, n + len(seen_items))
    top_n_scores, top_n_indices = torch.topk(dot_product, n)

    # Map indices back to item IDs
    top_n_item_ids = [list(item_id_to_idx.keys())[list(item_id_to_idx.values()).index(idx.item())] for idx in top_n_indices]
    # print(top_n_item_ids)
    # Filter out seen items
    # unseen_top_n_item_ids = [item for item in top_n_item_ids if item not in seen_items]
    # print(unseen_top_n_item_ids[:n])
    # return unseen_top_n_item_ids[:n]
    # print(top_n_item_ids[:n])
    return top_n_item_ids[:n]


In [91]:
tt_recommendations = {}

for user in test_ratings['user_id'].unique():
    # Generate top N recommendations for the user using the Two-Tower model
    recommended_items = get_top_n_items_without_history_unseen_items(best_model, user, n=5)
    tt_recommendations[user] = recommended_items  # Store the recommendations in the dictionary
tt_recommendations

{'AZUNT3QP2CWTL': ['B0007IT54C',
  'B00089RGEG',
  'B00085VVOG',
  'B000MYUN6K',
  'B000UD90BC'],
 'A3VVDE8I22IAJA': ['B0007KA6DO',
  'B000N7C8NC',
  'B000855VIS',
  'B000KW0HGK',
  '3438052180'],
 'A3AZ4O4I9S4668': ['B0007KA6DO',
  'B000JN3VWM',
  'B000KW1DPE',
  'B0008ADB58',
  'B000UD90BC'],
 'A319KYEIAZ3SON': ['B0006C2DE8',
  'B0006X79WI',
  'B0006AQJIQ',
  'B000K06SKQ',
  'B000QBQ24G'],
 'A3SOB0CMUBK6XJ': ['B000KYFTGQ',
  '019214183X',
  'B0007HQRJE',
  'B000859E8Q',
  '1841155683'],
 'A3927BH5H75LII': ['B0007HBBBI',
  'B000PMAOZ4',
  'B000N8SFVK',
  '1557503524',
  'B00085VVOG'],
 'A2FR8GG77M4TP7': ['B000MQAD4U',
  'B0007ETA7S',
  '1592286151',
  'B00086V1DG',
  'B000L90QOU'],
 'A2YUZKPLUYQDKV': ['061399860X',
  'B00005WKG3',
  'B000NX6AE4',
  '1552853705',
  '1565540565'],
 'A1HLJAZ1J5MMAB': ['B000K06SKQ',
  'B000QBQ24G',
  'B000FQ555Y',
  'B0007J4U78',
  'B000UD90BC'],
 'A1GBOCJ943SP8R': ['B000NX6AE4',
  'B000PS2WSK',
  '1883052432',
  'B00085L5R4',
  'B000H2TDYK'],
 'AXOA9OI96

In [92]:
def rrf_score(ranks, k=5):
    return sum([1 / (k + rank) for rank in ranks])

def combine_recommendations_with_rrf(user_id, cf_recommendations, tt_recommendations, k=5):
    # Create a dictionary to hold the RRF scores
    combined_scores = {}

    # Assign ranks and calculate RRF scores from Collaborative Filtering recommendations
    for rank, item in enumerate(cf_recommendations[user_id], start=1):
        if item not in combined_scores:
            combined_scores[item] = rrf_score([rank], k)
        else:
            combined_scores[item] += rrf_score([rank], k)

    # Assign ranks and calculate RRF scores from Two-Tower recommendations
    for rank, item in enumerate(tt_recommendations[user_id], start=1):
        if item not in combined_scores:
            combined_scores[item] = rrf_score([rank], k)
        else:
            combined_scores[item] += rrf_score([rank], k)

    # Sort the items based on their RRF scores in descending order
    sorted_items = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)

    # Return the top N items
    return [item for item, score in sorted_items[:k]]

In [93]:
final_recommendations = {}
for user_id in test_ratings['user_id'].unique():
    final_recommendations[user_id] = combine_recommendations_with_rrf(user_id, cf_recommendations, tt_recommendations, k=5)

# Now final_recommendations contains the combined top N recommendations for each user
final_recommendations

{'AZUNT3QP2CWTL': ['B000JMKVEO',
  'B0007IT54C',
  'B000CSYUNS',
  'B00089RGEG',
  'B0006DJTQM'],
 'A3VVDE8I22IAJA': ['8188280046',
  'B0007KA6DO',
  '1844560333',
  'B000N7C8NC',
  'B000NOX190'],
 'A3AZ4O4I9S4668': ['009133571X',
  'B0007KA6DO',
  'B000NA47QU',
  'B000JN3VWM',
  'B000HIIZXE'],
 'A319KYEIAZ3SON': ['B000P2A4B8',
  'B0006C2DE8',
  'B000MV8HQ6',
  'B0006X79WI',
  '1582010269'],
 'A3SOB0CMUBK6XJ': ['B000IEZE3G',
  'B000KYFTGQ',
  'B000MY6VHK',
  '019214183X',
  'B000FCB4K8'],
 'A3927BH5H75LII': ['B000F5X89K',
  'B0007HBBBI',
  'B000EANQJ8',
  'B000PMAOZ4',
  '1850891648'],
 'A2FR8GG77M4TP7': ['B0007DLN6K',
  'B000MQAD4U',
  'B0006IU3DU',
  'B0007ETA7S',
  'B0002Z0M4W'],
 'A2YUZKPLUYQDKV': ['B000U2I320',
  '061399860X',
  'B000NOXDUC',
  'B00005WKG3',
  'B000H7GW2Q'],
 'A1HLJAZ1J5MMAB': ['B000GR0EL2',
  'B000K06SKQ',
  '044108950X',
  'B000QBQ24G',
  'B000P6L6WU'],
 'A1GBOCJ943SP8R': ['B000OVTHX6',
  'B000NX6AE4',
  'B000KA1JCI',
  'B000PS2WSK',
  'B000HQXQEE'],
 'AXOA9OI96

In [107]:
final_recommendations

{'AZUNT3QP2CWTL': ['B000JMKVEO',
  'B000CSYUNS',
  'B0006DJTQM',
  'B0007IT54C',
  'B000GQJP2M'],
 'A3VVDE8I22IAJA': ['8188280046',
  '1844560333',
  'B000NOX190',
  'B0007KA6DO',
  'B0008CXTHG'],
 'A3AZ4O4I9S4668': ['009133571X',
  'B000NA47QU',
  'B000HIIZXE',
  'B0007KA6DO',
  'B000I2UNB6'],
 'A319KYEIAZ3SON': ['B000P2A4B8',
  'B000MV8HQ6',
  '1582010269',
  'B0006C2DE8',
  '1400132541'],
 'A3SOB0CMUBK6XJ': ['B000IEZE3G',
  'B000MY6VHK',
  'B000FCB4K8',
  'B000KYFTGQ',
  'B000NPEWHE'],
 'A3927BH5H75LII': ['B000F5X89K',
  'B000EANQJ8',
  '1850891648',
  'B0007HBBBI',
  'B000I3NFKG'],
 'A2FR8GG77M4TP7': ['B0007DLN6K',
  'B0006IU3DU',
  'B0002Z0M4W',
  'B000MQAD4U',
  'B000P0W8K0'],
 'A2YUZKPLUYQDKV': ['B000U2I320',
  'B000NOXDUC',
  'B000H7GW2Q',
  '061399860X',
  'B000HEGHT2'],
 'A1HLJAZ1J5MMAB': ['B000GR0EL2',
  '044108950X',
  'B000P6L6WU',
  'B000K06SKQ',
  'B000GLI9HY'],
 'A1GBOCJ943SP8R': ['B000OVTHX6',
  'B000KA1JCI',
  'B000HQXQEE',
  'B000NX6AE4',
  'B000GVE24S'],
 'AXOA9OI96

In [115]:
def dcg(scores, k):
    scores = np.asfarray(scores)[:k]
    return np.sum(scores / np.log2(np.arange(2, scores.size + 2)))

def ndcg_at_k(labels, k):
    ideal_labels = sorted(labels, reverse=True)
    return dcg(labels, k) / dcg(ideal_labels, k)

def recall_at_k(labels, relevant_count, k):
    return np.sum(labels[:k]) / relevant_count

def mrr_at_k(labels, k):
    for i, label in enumerate(labels[:k]):
        if label == 1:
            return 1 / (i + 1)
    return 0

def evaluate_final_recommendations(final_recommendations, test_ratings, k=5):
    ndcg_scores = []
    recall_scores = []
    mrr_scores = []

    # Get unique users
    unique_users = test_ratings['user_id'].unique()

    for user in unique_users:
        # Get the recommended items for the user
        recommended_items = final_recommendations[user]

        # Get the actual items the user interacted with in the test set
        user_test_data = test_ratings[test_ratings['user_id'] == user]
        test_items = user_test_data['item_id'].values

        # Create a binary relevance score list (1 if item is in the test set, 0 otherwise)
        y_score = [1 if item in test_items else 0 for item in recommended_items]

        # Calculate the evaluation metrics
        ndcg = ndcg_at_k(y_score, k)
        recall = recall_at_k(y_score, len(test_items), k)
        mrr = mrr_at_k(y_score, k)

        # Store the scores
        ndcg_scores.append(ndcg)
        recall_scores.append(recall)
        mrr_scores.append(mrr)

    # Compute average scores
    avg_ndcg = np.nanmean(ndcg_scores)
    avg_recall = np.nanmean(recall_scores)
    avg_mrr = np.nanmean(mrr_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg,
        'Recall@{}'.format(k): avg_recall,
        'MRR@{}'.format(k): avg_mrr,
    }

# Evaluate the final recommendations
# eval_result = evaluate_final_recommendations(final_recommendations, test_ratings, k=5)
# print(eval_result)
eval_result = evaluate_final_recommendations(final_recommendations, test_ratings, k=5)
print(eval_result)

C:\Users\Hooman\AppData\Local\Temp\ipykernel_11928\3737897827.py:7: RuntimeWarning: invalid value encountered in scalar divide
  return dcg(labels, k) / dcg(ideal_labels, k)


{'NDCG@5': 0.8610083496264046, 'Recall@5': 0.14189183602152267, 'MRR@5': 0.3732457528753825}


In [103]:
def evaluate_final_recommendations(final_recommendations, test_ratings, k=5):
    ndcg_scores = []
    recall_scores = []
    mrr_scores = []

    # Get unique users
    unique_users = test_ratings['user_id'].unique()

    for user in unique_users:
        # Get the recommended items for the user
        recommended_items = final_recommendations[user]

        # Get the actual items the user interacted with in the test set
        user_test_data = test_ratings[test_ratings['user_id'] == user]
        test_items = user_test_data['item_id'].values

        # Create a binary relevance score list (1 if item is in the test set, 0 otherwise)
        y_score = [
            user_test_data[user_test_data['item_id'] == item]['rating'].values[0] if item in test_items else 2.5
            for item in recommended_items
        ]
        # Calculate the evaluation metrics
        ndcg = ndcg_at_k(y_score, k)
        recall = recall_at_k(y_score, len(test_items), k)
        mrr = mrr_at_k(y_score, k)

        # Store the scores
        ndcg_scores.append(ndcg)
        recall_scores.append(recall)
        mrr_scores.append(mrr)

    # Compute average scores
    avg_ndcg = np.nanmean(ndcg_scores)
    avg_recall = np.nanmean(recall_scores)
    avg_mrr = np.nanmean(mrr_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg
    }

# Evaluate the final recommendations
eval_result = evaluate_final_recommendations(final_recommendations, test_ratings, k=5)
print(eval_result)
# eval_result = evaluate_final_recommendations(final_recommendations, test_ratings, k=10)
# print(eval_result)

{'NDCG@5': 0.9890279046788113}


In [104]:
def evaluate_final_recommendations(final_recommendations, test_ratings, k=5):
    ndcg_scores = []
    recall_scores = []
    mrr_scores = []

    # Get unique users
    unique_users = test_ratings['user_id'].unique()

    for user in unique_users:
        # Get the recommended items for the user
        recommended_items = final_recommendations[user]

        # Get the actual items the user interacted with in the test set
        user_test_data = test_ratings[test_ratings['user_id'] == user]
        test_items = user_test_data['item_id'].values

        # Create a binary relevance score list (1 if item is in the test set, 0 otherwise)
        y_score = [
            user_test_data[user_test_data['item_id'] == item]['rating'].values[0] if item in test_items else 0
            for item in recommended_items
        ]
        # Calculate the evaluation metrics
        ndcg = ndcg_at_k(y_score, k)
        recall = recall_at_k(y_score, len(test_items), k)
        mrr = mrr_at_k(y_score, k)

        # Store the scores
        ndcg_scores.append(ndcg)
        recall_scores.append(recall)
        mrr_scores.append(mrr)

    # Compute average scores
    avg_ndcg = np.nanmean(ndcg_scores)
    avg_recall = np.nanmean(recall_scores)
    avg_mrr = np.nanmean(mrr_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg,
    }

# Evaluate the final recommendations
eval_result = evaluate_final_recommendations(final_recommendations, test_ratings, k=5)
print(eval_result)
# eval_result = evaluate_final_recommendations(final_recommendations, test_ratings, k=10)
# print(eval_result)

C:\Users\Hooman\AppData\Local\Temp\ipykernel_11928\3737897827.py:7: RuntimeWarning: invalid value encountered in scalar divide
  return dcg(labels, k) / dcg(ideal_labels, k)


{'NDCG@5': 0.7718941107606238}


In [113]:
def rrf_score(ranks, k=5):
    return sum([1 / (k + rank) for rank in ranks])

def combine_recommendations_with_rrf(user_id, cf_recommendations, tt_recommendations, k=5, cf_weight=1, tt_weight=4):
    # Create a dictionary to hold the RRF scores
    combined_scores = {}

    # Assign ranks and calculate weighted RRF scores from Collaborative Filtering recommendations
    for rank, item in enumerate(cf_recommendations[user_id], start=1):
        weighted_rank = rank * cf_weight  # Apply the weight to CF ranks
        if item not in combined_scores:
            combined_scores[item] = rrf_score([weighted_rank], k)
        else:
            combined_scores[item] += rrf_score([weighted_rank], k)

    # Assign ranks and calculate weighted RRF scores from Two-Tower recommendations
    for rank, item in enumerate(tt_recommendations[user_id], start=1):
        weighted_rank = rank * tt_weight  # Apply the weight to TT ranks
        if item not in combined_scores:
            combined_scores[item] = rrf_score([weighted_rank], k)
        else:
            combined_scores[item] += rrf_score([weighted_rank], k)

    # Sort the items based on their RRF scores in descending order
    sorted_items = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)

    # Return the top N items
    return [item for item, score in sorted_items[:k]]

In [114]:
final_recommendations = {}
for user_id in test_ratings['user_id'].unique():
    final_recommendations[user_id] = combine_recommendations_with_rrf(user_id, cf_recommendations, tt_recommendations, k=5)

# Now final_recommendations contains the combined top N recommendations for each user
final_recommendations

{'AZUNT3QP2CWTL': ['B0007HUT02',
  'B000GKURY8',
  'B0007DRGI4',
  'B000N3ZGB2',
  'B0007IT54C'],
 'A3VVDE8I22IAJA': ['1844560333',
  '8188280046',
  '1593355548',
  '1569602093',
  'B0007KA6DO'],
 'A3AZ4O4I9S4668': ['B000QB9ZYA',
  'B0006AP72A',
  '1597374555',
  'B000K0H0OO',
  'B0007KA6DO'],
 'A319KYEIAZ3SON': ['1844560333',
  '1901768600',
  'B000HEGYW2',
  'B000RAZERW',
  'B0006C2DE8'],
 'A3SOB0CMUBK6XJ': ['B0006BV6RY',
  '1844560333',
  '8188280046',
  '1587263971',
  'B000KYFTGQ'],
 'A3927BH5H75LII': ['B000I3NFKG',
  'B000EANQJ8',
  'B0007C10MS',
  'B0007GZPJI',
  'B0007HBBBI'],
 'A2FR8GG77M4TP7': ['B000HKLROQ',
  'B00007K45C',
  'B000P0W8K0',
  'B000TKO3EA',
  'B000MQAD4U'],
 'A2YUZKPLUYQDKV': ['B000L4056E',
  'B000NDSX6C',
  'B000Q032UY',
  'B000ILIJE0',
  '061399860X'],
 'A1HLJAZ1J5MMAB': ['B000KXXBXA',
  '1593555164',
  'B00005W9E7',
  'B000K6TBBS',
  'B000K06SKQ'],
 'A1GBOCJ943SP8R': ['B000QAA98W',
  'B000L1RWA4',
  'B000JWXXEY',
  'B000OVDPPW',
  'B000NX6AE4'],
 'AXOA9OI96